# ReWOO vs ReAct · token 消耗对比

**任务**：多跳问答。先用一个迷你 KB（沿用第 04 章），构造若干 *2-3 跳* 的问题。

对比：
- **ReAct**：Anthropic tool use，多轮调用，每轮把全 history 发给大 LLM。
- **ReWOO**：Planner（LLM, 1 次）→ Workers（工具）→ Solver（LLM, 1 次）。

重点观察：token 总消耗 / 准确率 / 失败模式。

In [ ]:
import os, sys, re, json, numpy as np
sys.path.append(os.path.abspath('../..'))
from utils.llm_client import LLMClient
from anthropic import Anthropic
client = LLMClient(temperature=0)
anthropic = Anthropic()
MODEL = client.model

## 1. 工具：search_docs（与第 04 章相同）

In [ ]:
from sentence_transformers import SentenceTransformer
_st = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
DOCS = [
    'ReAct（Yao 2022）让 LLM 交替输出 Thought/Action/Observation。',
    'Reflexion（Shinn 2023）通过自然语言反思在多 episode 间积累经验。',
    'Tree of Thoughts（Yao 2023）把 CoT 思路扩展为搜索树。',
    'LATS（Zhou 2024）= ToT + ReAct + Reflexion，用 MCTS 串起搜索/行动/反思。',
    'Generative Agents（Park 2023）三层记忆：观察、反思、计划。',
    'MemGPT（Packer 2023）借 OS 虚拟内存思想管理 working/archival memory。',
    'Self-RAG（Asai 2024）让 LLM 输出 reflection token 决定检索/引用。',
    'Corrective RAG（CRAG, Yan 2024）检索失败时转 web 搜索兜底。',
    'MCP（Anthropic 2024）把 LLM ↔ 工具/数据接口标准化。',
    'CodeAct（Wang 2024）用可执行 Python 作为统一 action 表达。',
    'AutoGen（Microsoft 2023）以多 agent 对话为核心抽象。',
    'MetaGPT（Hong 2023）多角色 agent 模拟软件公司 SOP。',
    'GAIA（Mialon 2023）通用 agent benchmark，强调多步推理 + 工具。',
    'SWE-bench（Jimenez 2023）从真实 GitHub issue 构造修复测试。',
    'WebArena（Zhou 2023）可重现 web 任务环境。',
    'GRPO（DeepSeek 2024）省掉 value model，用组内归一 advantage。',
    'DAPO（ByteDance 2025）GRPO 改进版，AIME 2024 达 50 分。',
    'SkyRL-Agent（2025）多轮长程 agent RL 训练框架。',
    'MapAgent（2025）分层多 agent 框架，map-tool agent 并行调地图 API。',
    'PReP（2024）perceive-reflect-plan 三阶段做无指令城市导航。',
]
EMB = _st.encode(DOCS, normalize_embeddings=True)

def search_docs(query: str, k: int = 3):
    q = _st.encode([query], normalize_embeddings=True)[0]
    sims = EMB @ q
    idx = np.argsort(-sims)[:k]
    return [DOCS[j] for j in idx]

## 2. ReAct 版（Anthropic tool use）

In [ ]:
TOOLS = [{
    'name': 'search_docs',
    'description': '检索 LLM agent 知识库片段。',
    'input_schema': {
        'type': 'object',
        'properties': {'query': {'type': 'string'}, 'k': {'type': 'integer', 'default': 3}},
        'required': ['query'],
    },
}]

def react_solve(question: str, max_steps: int = 6):
    messages = [{'role': 'user', 'content': question}]
    in_tok = out_tok = 0
    for _ in range(max_steps):
        resp = anthropic.messages.create(model=MODEL, max_tokens=512, tools=TOOLS, messages=messages)
        in_tok += resp.usage.input_tokens
        out_tok += resp.usage.output_tokens
        if resp.stop_reason != 'tool_use':
            ans = ''.join(b.text for b in resp.content if b.type == 'text')
            return ans, in_tok, out_tok
        messages.append({'role': 'assistant', 'content': resp.content})
        results = []
        for b in resp.content:
            if b.type == 'tool_use':
                hits = search_docs(**b.input)
                results.append({'type': 'tool_result', 'tool_use_id': b.id,
                                'content': '\n'.join(hits)})
        messages.append({'role': 'user', 'content': results})
    return '[max_steps]', in_tok, out_tok

## 3. ReWOO 版

实现简化版：
- Planner：让 LLM 输出形如 `\nPlan: ...\n#E1 = search_docs[query]\nPlan: ...\n#E2 = search_docs[#E1 ...]\n` 的纯文本。
- Worker：Python 解析 `#E1 / #E2 ...` 并执行；前面变量结果会被替换进后面的 query。
- Solver：把所有 evidence + 原 query 喂给 LLM 一次性写答。

In [ ]:
PLANNER_PROMPT = (
    '请把下面问题分解为可独立执行的检索步骤，使用变量占位 #E1, #E2, ...\n'
    '可用工具：search_docs[查询]，返回最相关的 3 个段落字符串。\n'
    '严格按以下格式输出，不要解释：\n'
    'Plan: <第 1 步描述>\n#E1 = search_docs[<查询>]\n'
    'Plan: <第 2 步描述>\n#E2 = search_docs[<查询，可以引用 #E1>]\n'
    '...\n\n问题：{q}\n'
)

STEP_RE = re.compile(r'Plan:\s*(?P<plan>[^\n]+)\n#E(?P<idx>\d+)\s*=\s*search_docs\[(?P<arg>[^\]]+)\]')
VAR_RE = re.compile(r'#E(\d+)')

def rewoo_solve(question: str):
    in_tok = out_tok = 0

    # 1) Planner
    p_msg = [{'role': 'user', 'content': PLANNER_PROMPT.format(q=question)}]
    p_resp = anthropic.messages.create(model=MODEL, max_tokens=512, messages=p_msg)
    in_tok += p_resp.usage.input_tokens
    out_tok += p_resp.usage.output_tokens
    plan_text = ''.join(b.text for b in p_resp.content if b.type == 'text')

    # 2) Workers
    evidences = {}
    for m in STEP_RE.finditer(plan_text):
        idx, arg = m.group('idx'), m.group('arg')
        for v in VAR_RE.findall(arg):
            arg = arg.replace(f'#E{v}', evidences.get(v, ''))
        evidences[idx] = '\n'.join(search_docs(arg, k=3))

    if not evidences:
        return f'[planner-failed]\n{plan_text}', in_tok, out_tok

    # 3) Solver
    ev_block = '\n'.join(f'#E{k}: {v}' for k, v in evidences.items())
    s_prompt = (
        '基于下面证据回答问题，请引用形如 [#E1] 标注。\n\n'
        f'证据：\n{ev_block}\n\n问题：{question}\n请简洁回答（≤80 字）：'
    )
    s_resp = anthropic.messages.create(model=MODEL, max_tokens=400,
                                       messages=[{'role': 'user', 'content': s_prompt}])
    in_tok += s_resp.usage.input_tokens
    out_tok += s_resp.usage.output_tokens
    ans = ''.join(b.text for b in s_resp.content if b.type == 'text')
    return ans, in_tok, out_tok

## 4. 多跳测试集

In [ ]:
QS = [
    '把 ToT、ReAct、Reflexion 三者结合的工作叫什么？它的搜索算法是什么？',
    '哪个 RAG 工作在检索失败时调 web 搜索？哪个让 LLM 生成 reflection token 决定检索？',
    '哪个 RL 算法不需要 value model？它的改进版是什么？',
    'Generative Agents 的三层记忆是什么？这种思想被哪个 OS 类比框架借鉴？',
]

rows = []
for q in QS:
    a_react, in_r, out_r = react_solve(q)
    a_rewoo, in_w, out_w = rewoo_solve(q)
    rows.append((q, a_react, in_r, out_r, a_rewoo, in_w, out_w))
    print('Q:', q)
    print(f'  ReAct ({in_r}+{out_r}={in_r+out_r} tok):', a_react)
    print(f'  ReWOO ({in_w}+{out_w}={in_w+out_w} tok):', a_rewoo)
    print('---')

In [ ]:
import statistics
react_tot = [r[2] + r[3] for r in rows]
rewoo_tot = [r[5] + r[6] for r in rows]
print(f'ReAct 平均 token: {statistics.mean(react_tot):.0f}')
print(f'ReWOO 平均 token: {statistics.mean(rewoo_tot):.0f}')
print(f'ReWOO 节省: {(1 - statistics.mean(rewoo_tot)/statistics.mean(react_tot))*100:.1f}%')

## 5. 思考

- ReWOO 把「Planner 规划 + Worker 执行 + Solver 总结」拆开，每个角色看到的上下文都更短。
- 在 *计划能事先确定* 的任务上节省可观；但若任务高度动态（中间结果决定后续步骤），ReWOO 的优势消失甚至反吃亏。
- 工程权衡：先用 LLM 评估任务可前置规划程度，再决定 ReAct vs ReWOO。

## 进阶练习

1. 让 Planner 输出严格 JSON，用 pydantic 校验，提升健壮性。
2. 加 *retry on failure*：Solver 觉得证据不足时回到 Planner 阶段补一步。
3. 试着让 Worker 之间并行执行（asyncio），观察延迟下降。